
To connect Azure Data Lake Storage Gen2 (ADLS) to Azure Databricks using a Databricks Access Connector, the recommended approach is to use Managed Identity + Unity Catalog Storage Credentials + External Locations.

![image_1792255489780.png](./image_1792255489780.png "image_1792255489780.png")

Step 1: Create the Access Connector in Azure
1. Open the Azure Portal.
2. Search for Access Connector for Azure Databricks.
3. Click Create.
4. Select the subscription, resource group, and region.
5. Under Managed Identity, enable System-assigned.
6. Click Review + create, then Create.
7. Open the connector and copy its Resource ID.

Step 2: Grant access to ADLS Gen2
Open your ADLS storage account in Azure Portal.
1. Go to Access Control (IAM).
2. Click Add → Add role assignment.
3. Select the role Storage Blob Data Contributor.
4. Under Members, select Managed identity.
5. Select Access Connector for Azure Databricks.
6. Choose your connector and click Review + assign.
This grants the managed identity read/write access to the storage account at the assigned scope. For production, prefer the narrowest scope that meets your requirements.



Step 3: Create a Storage Credential in Databricks
In your Azure Databricks workspace:
1. Open Catalog.
2. Click Create → Credential (the exact menu wording may vary).
3. Select credential type Azure Managed Identity.
4. Enter a name, such as adls_access_credential.
5. Paste the Access Connector's Resource ID.
6. If you use a user-assigned managed identity, also provide its resource ID.
7. Click Create.
You need a Unity Catalog-enabled workspace and permission to create storage credentials on the metastore.

Step 4: Create an External Location using SQL
In a Databricks SQL notebook, run:


CREATE EXTERNAL LOCATION IF NOT EXISTS adls_external_location
URL 'abfss://mycontainer@mystorageaccount.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adls_access_credential);

Replace:
- mycontainer with your ADLS container name.
- mystorageaccount with your storage account name.
- adls_access_credential with the storage credential you created.
The abfss:// scheme uses secure access to ADLS Gen2. The external location connects the storage path to the Unity Catalog storage credential.

### Step 5: Test the connection
Run this in a Python notebook attached to supported Databricks compute:

In [0]:
dbutils.fs.ls(f"abfss://rawdata@pndfinance.dfs.core.windows.net/")

### To read a CSV file:

In [0]:
df = spark.read.format("csv").option("header","true").load(f"abfss://rawdata@pndfinance.dfs.core.windows.net/account_1.csv")
display(df)